# T4 — DVC: versionamento de dados

**Objetivo:** versionar dados como versionamos código, sem colocar os dados no Git, e ligar cada modelo
à versão exata dos dados que o treinou.

| Peça | Onde fica | Vai para o Git? |
|---|---|---|
| `bike.csv` (o dado) | sua pasta + cache `.dvc/cache` + **remote** | ❌ |
| `bike.csv.dvc` (ponteiro: hash md5 + tamanho) | ao lado do dado | ✅ |
| `.dvc/config` (onde fica o remote) | `.dvc/` | ✅ |

> 💡 O DVC e o remote (`../dvc-storage`) foram configurados na T1. Lá, o pipeline só **anotava o hash** do
> `bike.csv` no `dvc.lock`. Agora o próprio dado passa a ser versionado.
>
> Os comandos que mudam o Git e o DVC vêm **comentados** (com `#`), para não rodarem sem querer.
> Quando chegar a hora, tire o `#` (selecione as linhas e use **Ctrl + /**) e rode a célula.

In [ ]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

---
# Parte 1 · Juntos

In [ ]:
%%bash
git status --short
git log --oneline | head -5
dvc remote list

## 1. Versionar os dados v1 (2011)

São sempre os mesmos 4 passos. **Guarde esta receita: você vai repeti-la na Parte 2.**

| Passo | Comando | O que faz |
|---|---|---|
| 1 | `dvc add data/raw/bike.csv` | calcula o md5, guarda no cache e cria o ponteiro `.dvc` |
| 2 | `git add data/raw/bike.csv.dvc` + `git commit` | o **ponteiro** vai para o Git |
| 3 | `git tag dados-v1` | um nome fácil para esta versão |
| 4 | `dvc push` | o **dado** vai para o remote |

In [ ]:
%%bash
# dvc add data/raw/bike.csv
cat data/raw/bike.csv.dvc

O `.gitignore` do template já ignora CSVs dentro de `data/`, por isso o DVC não precisou criar um novo.

In [ ]:
%%bash
# git add data/raw/bike.csv.dvc
# git commit -m "Dados v1: 2011"
# git tag dados-v1
# dvc push

## 2. Desastre: apagaram o dado **e** o cache

Se o dado está no remote, basta um `dvc pull`.

In [ ]:
%%bash
# rm -rf data/raw/bike.csv .dvc/cache
# dvc pull
wc -l data/raw/bike.csv

## 3. Como voltar no tempo (você vai usar na Parte 2)

Para trazer uma versão antiga são **2 comandos**: o Git traz o **ponteiro** antigo e o
`dvc checkout` traz o **dado** correspondente.

```bash
git checkout <tag> -- data/raw/bike.csv.dvc   # ponteiro da versão <tag>
dvc checkout                                  # dado que corresponde ao ponteiro
```
Para voltar à versão atual, troque `<tag>` por `HEAD`.

---
# Parte 2 · Sozinhos

Chegaram dados novos: o **1º semestre de 2012**. Versione-os com a mesma receita da Parte 1.
Repare no `dvc status`: ele mostra que o dado mudou **e** que o estágio `prepare` do pipeline, que depende dele, ficou desatualizado.

In [ ]:
%%bash
# python src/get_data.py --until 2012-06-30
dvc status

### 1. Versione os dados v2 (receita do passo 1)

In [ ]:
%%bash
# TODO: os 4 passos da receita, agora com a tag dados-v2 e a mensagem "Dados v2: até jun/2012"
git log --oneline --decorate | head -5

### 2. Viagem no tempo: volte para a v1, conte as linhas e retorne para a v2 (passo 3)

In [ ]:
%%bash
# TODO: traga a versão dados-v1 (2 comandos)
wc -l data/raw/bike.csv      # v1: 8.646 linhas (8.645 + cabeçalho)

In [ ]:
%%bash
# TODO: volte para a versão atual (HEAD)
wc -l data/raw/bike.csv      # v2: 13.004 linhas

### 3. Retreine o pipeline com os dados v2 (o `dvc repro` da T1)

Como o dado mudou, o pipeline inteiro roda de novo. O treino gera um novo run no MLflow e uma nova versão no Registry.

In [ ]:
%%bash
# TODO: rode o pipeline e mostre as métricas (dvc metrics show)

### 4. Guarde o recibo e publique no GitHub: o código, os ponteiros e as tags (os dados não vão)

In [ ]:
%%bash
# TODO: commit de dvc.lock, metrics.json e reports/previsto_vs_real.png; depois dvc push, git push e git push --tags